In [3]:
# =============================================================================
# Fed Direction Dashboard
# Pulls hard-number macro series from FRED, and pulls the "no clean API" series
# (ISM PMIs, Chicago PMI, Consumer Confidence) from an LLM w/ web search,
# querying 3x and averaging to reduce single-call hallucination/staleness risk.
#
# ENV VARS REQUIRED:
#   FRED_API_KEY   - https://fred.stlouisfed.org/docs/api/api_key.html
#   OPENAI_API_KEY - standard OpenAI key (picked up automatically by the SDK)
#
# pip install requests pandas openai
# =============================================================================

import os
import re
import statistics
import requests
import pandas as pd
from datetime import datetime
from openai import OpenAI
from IPython.display import display

FRED_API_KEY = os.environ["FRED_API_KEY"]
client = OpenAI()  # reads OPENAI_API_KEY from env automatically

# -----------------------------------------------------------------------------
# 1. FRED helpers
# -----------------------------------------------------------------------------

def fred_observations(series_id, limit=14):
    """Return list of (date, float_value), most recent first, '.' (missing) dropped."""
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "sort_order": "desc",
        "limit": limit,
    }
    r = requests.get(url, params=params, timeout=15)
    r.raise_for_status()
    data = r.json()["observations"]
    return [(o["date"], float(o["value"])) for o in data if o["value"] != "."]


def fred_latest_level(series_id):
    obs = fred_observations(series_id, limit=1)
    return obs[0][1], obs[0][0]


def fred_mom_change_thousands(series_id):
    """Level series (already in thousands, e.g. PAYEMS) -> most recent MoM diff."""
    obs = fred_observations(series_id, limit=2)
    latest, prev = obs[0][1], obs[1][1]
    return latest - prev, obs[0][0]


def fred_mom_pct_change(series_id):
    obs = fred_observations(series_id, limit=2)
    latest, prev = obs[0][1], obs[1][1]
    pct = (latest / prev - 1) * 100
    return pct, obs[0][0]


def fred_yoy_pct_change(series_id):
    obs = fred_observations(series_id, limit=14)
    latest_date, latest_val = obs[0]
    year_ago_val = obs[12][1]  # 12 months prior, monthly series
    pct = (latest_val / year_ago_val - 1) * 100
    return pct, latest_date


# -----------------------------------------------------------------------------
# 2. LLM (web-search) helper for metrics with no free API
#    Called 3x per metric, numeric answers averaged.
# -----------------------------------------------------------------------------

def llm_metric_value(metric_label, question, n_calls=3, model="gpt-4o"):
    values = []
    raw_texts = []
    for _ in range(n_calls):
        resp = client.responses.create(
            model=model,
            tools=[{"type": "web_search_preview"}],
            input=(
                f"{question} "
                "Reply with ONLY the single most recent published numeric value "
                "(no % sign, no commentary, no units, no date) — just the number."
            ),
        )
        text = resp.output_text.strip()
        raw_texts.append(text)
        match = re.search(r"-?\d+\.?\d*", text)
        if match:
            values.append(float(match.group()))
    if not values:
        raise ValueError(f"Could not parse a numeric value for {metric_label}: {raw_texts}")
    return statistics.mean(values), values, raw_texts


LLM_QUERIES = {
    "ISM Manufacturing PMI": "What is the latest ISM Manufacturing PMI (Purchasing Managers' Index) reading?",
    "ISM Non-Manufacturing PMI": "What is the latest ISM Non-Manufacturing (Services) PMI reading?",
    "Chicago PMI": "What is the latest Chicago Business Barometer (Chicago PMI) reading?",
    "Consumer Confidence": "What is the latest Conference Board Consumer Confidence Index value?",
}

# -----------------------------------------------------------------------------
# 3. Classification rules -> (state, fed_action)
# -----------------------------------------------------------------------------

def classify_unemployment(v):
    if v > 5.5: return "High", "Expansionary"
    if v < 4.0: return "Low", "Contractionary"
    return "Normal", "Neutral"

def classify_jobless_claims(v):
    if v > 350_000: return "High", "Expansionary"
    if v < 250_000: return "Low", "Contractionary"
    return "Normal", "Neutral"

def classify_nonfarm_payrolls(v):  # v in thousands
    if v > 250: return "High", "Contractionary"
    if v < 50: return "Low", "Expansionary"
    return "Normal", "Neutral"

def classify_inflation_pct(v):  # CPI YoY, Core CPI YoY
    if v > 2.2: return "High", "Contractionary"
    if v < 1.8: return "Low", "Expansionary"
    return "Normal", "Neutral"

def classify_pce_yoy(v):
    if v > 2.5: return "High", "Contractionary"
    if v < 1.8: return "Low", "Expansionary"
    return "Normal", "Neutral"

def classify_core_pce_yoy(v):
    if v > 2.2: return "High", "Contractionary"
    if v < 1.8: return "Low", "Expansionary"
    return "Normal", "Neutral"

def classify_ppi_mom(v):
    if v > 0.2: return "High", "Contractionary"
    if v < 0.0: return "Low", "Expansionary"
    return "Normal", "Neutral"

def classify_pmi(v):
    if v > 55: return "High", "Contractionary"
    if v < 50: return "Low", "Expansionary"
    return "Normal", "Neutral"

def classify_consumer_confidence(v):
    if v > 120: return "High", "Contractionary"
    if v < 100: return "Low", "Expansionary"
    return "Normal", "Neutral"


# -----------------------------------------------------------------------------
# 4. Pull everything
# -----------------------------------------------------------------------------

rows = []

# --- FRED-backed metrics ---
v, d = fred_latest_level("UNRATE")
state, action = classify_unemployment(v)
rows.append(["Unemployment Rate", f"{v:.1f}%", d, state, action, "FRED"])

v, d = fred_latest_level("ICSA")
state, action = classify_jobless_claims(v)
rows.append(["Initial Jobless Claims", f"{v:,.0f}", d, state, action, "FRED"])

v, d = fred_mom_change_thousands("PAYEMS")
state, action = classify_nonfarm_payrolls(v)
rows.append(["Nonfarm Payrolls (MoM chg)", f"{v:,.0f}k", d, state, action, "FRED"])

v, d = fred_yoy_pct_change("CPIAUCSL")
state, action = classify_inflation_pct(v)
rows.append(["CPI (YoY)", f"{v:.2f}%", d, state, action, "FRED"])

v, d = fred_yoy_pct_change("CPILFESL")
state, action = classify_inflation_pct(v)
rows.append(["Core CPI (YoY)", f"{v:.2f}%", d, state, action, "FRED"])

v, d = fred_yoy_pct_change("PCEPI")
state, action = classify_pce_yoy(v)
rows.append(["PCE (YoY) - headline", f"{v:.2f}%", d, state, action, "FRED"])

v, d = fred_yoy_pct_change("PCEPILFE")
state, action = classify_core_pce_yoy(v)
rows.append(["Core PCE (YoY)", f"{v:.2f}%", d, state, action, "FRED"])

v, d = fred_mom_pct_change("PPIACO")
state, action = classify_ppi_mom(v)
rows.append(["PPI (MoM)", f"{v:.2f}%", d, state, action, "FRED"])

# --- LLM (web search) backed metrics, 3 calls averaged ---
for label, question in LLM_QUERIES.items():
    avg, samples, _ = llm_metric_value(label, question)
    if label == "Consumer Confidence":
        state, action = classify_consumer_confidence(avg)
    else:
        state, action = classify_pmi(avg)
    rows.append([label, f"{avg:.1f}", f"avg of {samples}", state, action, "LLM (web search x3)"])

df = pd.DataFrame(rows, columns=["Metric", "Value", "As Of / Detail", "State", "Fed Action", "Source"])

# -----------------------------------------------------------------------------
# 5. Color-coded display
# -----------------------------------------------------------------------------

STATE_COLORS = {"High": "#f8d7da", "Normal": "#fff3cd", "Low": "#d4edda"}
ACTION_COLORS = {"Expansionary": "#d4edda", "Neutral": "#fff3cd", "Contractionary": "#f8d7da"}

def style_row(row):
    styles = [""] * len(row)
    state_idx = df.columns.get_loc("State")
    action_idx = df.columns.get_loc("Fed Action")
    styles[state_idx] = f"background-color: {STATE_COLORS.get(row['State'], '')}"
    styles[action_idx] = f"background-color: {ACTION_COLORS.get(row['Fed Action'], '')}"
    return styles

styled = df.style.apply(style_row, axis=1).set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")] }]
)
display(styled)

# -----------------------------------------------------------------------------
# 6. Summary: counts + overall Fed-direction lean
# -----------------------------------------------------------------------------

counts = df["Fed Action"].value_counts().reindex(
    ["Expansionary", "Neutral", "Contractionary"], fill_value=0
)

print("\n=== Fed Direction Summary (as of {}) ===".format(datetime.now().strftime("%Y-%m-%d")))
for action, cnt in counts.items():
    bar = "█" * cnt
    print(f"{action:<15} {cnt:>2}  {bar}")

lean = counts.idxmax()
print(f"\nOverall lean based on majority of {len(df)} metrics: **{lean}**")

,Metric,Value,As Of / Detail,State,Fed Action,Source
0,Unemployment Rate,4.2%,2026-06-01,Normal,Neutral,FRED
1,Initial Jobless Claims,"215,000",2026-06-27,Low,Contractionary,FRED
2,Nonfarm Payrolls (MoM chg),57k,2026-06-01,Normal,Neutral,FRED
3,CPI (YoY),4.27%,2026-05-01,High,Contractionary,FRED
4,Core CPI (YoY),2.96%,2026-05-01,High,Contractionary,FRED
5,PCE (YoY) - headline,4.07%,2026-05-01,High,Contractionary,FRED
6,Core PCE (YoY),3.41%,2026-05-01,High,Contractionary,FRED
7,PPI (MoM),3.47%,2026-05-01,High,Contractionary,FRED
8,ISM Manufacturing PMI,53.3,"avg of [53.3, 53.3, 53.3]",Normal,Neutral,LLM (web search x3)
9,ISM Non-Manufacturing PMI,54.5,"avg of [54.5, 54.5, 54.5]",Normal,Neutral,LLM (web search x3)



=== Fed Direction Summary (as of 2026-07-04) ===
Expansionary     1  █
Neutral          4  ████
Contractionary   7  ███████

Overall lean based on majority of 12 metrics: **Contractionary**
